# The Catalyst Optimizer: Rule-Based and Cost-Based Optimizations

**Module**: 07_spark_sql_internals  
**Notebook**: 01_catalyst_optimizer.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Catalyst rewrites tree structures using rule-based transformations (Predicate Pushdown, Column Pruning, Constant Folding) and Cost-Based Optimization (CBO).
- **Why does this topic exist?**: Catalyst is the query optimizer that makes Spark SQL and DataFrames fast.
- **What problem does it solve?**: Understanding how Spark optimizes your queries behind the scenes.
- **Where is it used in real-world Data Engineering?**: Explaining query optimization in senior data engineering interviews.

---

## 2. Catalyst Tree Transformation Pipeline
1. **Analysis**: Resolves column names against Metastore Catalog.
2. **Logical Optimization**: Applies rule-based pattern matching:
   - `PushDownPredicate`: Pushes filters closer to the data source.
   - `ColumnPruning`: Discards unneeded columns early.
   - `ConstantFolding`: Evaluates constant expressions at compile time (`1 + 1 -> 2`).
3. **Physical Planning**: Cost model selects algorithms (e.g. BroadcastHashJoin vs SortMergeJoin).
4. **Code Generation**: Tungsten compiles plan into Java bytecode.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Catalyst_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
# Example query illustrating Constant Folding and Filter Pushdown
q = orders.filter("quantity > (1 + 1)").select("order_id", "status")
print("=== Optimized Logical Plan showing Constant Folding ===")
q.explain(True)

---

## Experiment: Constant Folding Demonstration

- **1. What are we testing?**: Verify in the Optimized Logical Plan that (1 + 1) was evaluated to 2 before execution.
- **2. What do we expect?**: Optimized plan shows GreaterThan(quantity, 2).

In [ ]:
print(q._jdf.queryExecution().optimizedPlan().toString())

**3. What happened?**: Expression was reduced to constant 2.  
**4. Why did it happen?**: Catalyst rule ConstantFolding evaluated the literal arithmetic.  
**5. Practical Engineering Lesson**: Spark does not evaluate arithmetic expressions row-by-row at runtime if they can be evaluated once at compile time.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Column Pruning
**Problem Statement**: Verify that unused columns in orders are pruned in the physical plan.  
**Input / Context**: orders.select('order_id')

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Combine Filters Rule
**Problem Statement**: Observe that two consecutive filter() calls are combined into one AND expression.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost-Based Optimizer Flag
**Problem Statement**: Print spark.sql.cbo.enabled setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Column Pruning):
```python
orders.select('order_id').explain()
```

#### Solution 2 (Combine Filters Rule):
```python
orders.filter('quantity > 1').filter('quantity < 10').explain()
```

#### Solution 3 (Cost-Based Optimizer Flag):
```python
print(spark.conf.get('spark.sql.cbo.enabled','false'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Catalyst Power**: Tree transformation framework.
- **Rules**: Predicate pushdown, column pruning, constant folding.

### Essential Technical Terms
- `Catalyst`: Extensible Scala query optimizer.
- `Rule[LogicalPlan]`: Pattern-matching optimization rule.

### Common Engineering Mistakes
- **Over-optimizing manually**: Catalyst already reorders filters and prunes columns.

### Interview Questions on This Topic
- **Q: What are the primary phases of the Catalyst optimizer?**
  - *A*: Analysis, Logical Optimization, Physical Planning, and Code Generation.

### What to Study Next
Open 02_logical_and_physical_plans.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')